In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
# 超参数设置
batch_size = 64
block_size = 256


# 数据集获取、处理
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))  # 以字符作为 token 
vocab_size = len(chars)          # 根据字符获取字典大小


# 构建字符到索引的映射，encoder、decoder
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]           # 字符    -> 字典索引
decode = lambda l: ''.join([itos[i] for i in l])  # 字典索引 -> 字符


# 区分训练集和测试集
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


# 辅助函数
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
class Generate():
    @torch.no_grad()  # 生成文本时不需要计算梯度，会节省内存和计算资源
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            # 获取输入的最后 block_size 个字符
            idx_cond = idx[:, -block_size:]
            # 前向传播调用 forward()，获取下一个字符的概率分布
            logits, _ = self(idx_cond)
            # 获取最后一个时间步的输出
            logits = logits[:, -1, :]
            # 将 logits 转换为概率分布
            probs = F.softmax(logits / temperature, dim=-1)
            # 从概率分布中采样下一个字符的索引
            next_idx = torch.multinomial(probs, num_samples=1)
            # 将新生成的字符索引添加到输入序列中
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

class BigramLanguageModel(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

d_model = 32
class NanoGPTv1(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        # token_idx -> emb_idx
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        # position -> emb_pos
        self.pos_emb_table = nn.Embedding(block_size, d_model)

        # Language Model Head
        # 可以理解为 Un-embedding，将 emb_vec 转换回 token_idx
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        # 取前面所有的 emb_vec 的平均
        W_f3 = torch.tril(torch.ones((T, T), device=device))  # (T, T) 下三角矩阵
        W_f3 = W_f3 / torch.sum(W_f3, dim=1, keepdim=True)
        x = W_f3 @ x # (B, T, d_model)

        """
        # 加入 Softmax
        # 构造 W_f4：下三角为 0，上三角为 -inf
        tril = torch.tril(torch.ones((T, T), device=device))
        W_f4 = torch.zeros((T, T), device=device)
        W_f4 = W_f4.masked_fill(tril == 0, float('-inf'))
        # Softmax(W_f4) = W_f3
        W_f3 = F.softmax(W_f4, dim=1)
        x = W_f3 @ x  # (B, T, d_model)
        """

        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss


d_model = 32
class NanoGPTv2(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        # token_idx -> emb_idx
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        # position -> emb_pos
        self.pos_emb_table = nn.Embedding(block_size, d_model)

        self.W_f5 = nn.Parameter(torch.tril(torch.ones((block_size, block_size), device=device)))  # (T, T) 下三角矩阵
        # Language Model Head
        # 可以理解为 Un-embedding，将 emb_vec 转换回 token_idx
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        tril = torch.tril(torch.ones((T, T), device=device))
        wei = self.W_f5[:T, :T].masked_fill(tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=1)
        x = wei @ x  # (B, T, d_model)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

d_model = 384
d_k = 64
d_v = 64
class NanoGPTv3(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        self.pos_emb_table = nn.Embedding(block_size, d_model)
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        self.lm_head = nn.Linear(d_v, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        Q = self.W_Q(x)  # (B, T, d_k)
        K = self.W_K(x)  # (B, T, d_k)
        V = self.W_V(x)  # (B, T, d_v)
        W = Q @ K.transpose(-2, -1) * (d_k ** -0.5)
        
        tril = torch.tril(torch.ones((T, T), device=device))
        W = W.masked_fill(tril == 0, float('-inf'))
        W = F.softmax(W, dim=-1)

        y = W @ V  # (B, T, d_v)
        logits = self.lm_head(y)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss


d_model = 384
d_k = 64
d_v = 64

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        Q = self.W_Q(x)  # (B, T, d_k)
        K = self.W_K(x)  # (B, T, d_k)
        V = self.W_V(x)  # (B, T, d_v)
        W = Q @ K.transpose(-2, -1) * (d_k ** -0.5)
        
        tril = torch.tril(torch.ones((T, T), device=device))
        W = W.masked_fill(tril == 0, float('-inf'))
        W = F.softmax(W, dim=-1)

        y = W @ V  # (B, T, d_v)
        return y

class MultiHead(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * d_v, d_model)

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]  # List of (B, T, d_v) * num_heads
        concatenated = torch.cat(head_outputs, dim=-1)  # (B, T, num_heads * d_v) == (B, T, d_model)
        output = self.proj(concatenated)  # (B, T, d_model)
        return output

class NanoGPTv4(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        self.pos_emb_table = nn.Embedding(block_size, d_model)
        self.head = MultiHead(num_heads = d_model // d_v)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        y = self.head(x)  # (B, T, d_model)
        logits = self.lm_head(y)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

d_ff = 4 * d_model
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

class NanoGPTv5(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        self.pos_emb_table = nn.Embedding(block_size, d_model)
        self.heads = MultiHead(num_heads = d_model // d_v)
        self.ffn = FFN(d_model, d_ff)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        y = self.heads(x)  # (B, T, d_model)
        y = self.ffn(y)  # (B, T, d_model)
        logits = self.lm_head(y)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.heads = MultiHead(num_heads = d_model // d_v)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.dropout(self.heads(self.ln1(x))) + x  # (B, T, d_model)
        x = self.dropout(self.ffn(self.ln2(x))) + x     # (B, T, d_model)
        return x

num_blocks = 4
class NanoGPTv6(nn.Module, Generate):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb_table = nn.Embedding(vocab_size, d_model)
        self.pos_emb_table = nn.Embedding(block_size, d_model)
         
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb_table(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        x = self.blocks(x)  # (B, T, d_model)
        x = self.ln_f(x)  # (B, T, d_model)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
model = NanoGPTv6(vocab_size).to(device)

In [4]:
with torch.no_grad():
    input = "oh hi"
    input_encoded = encode(input)
    print(input_encoded)
    input_encoded = torch.tensor(input_encoded, dtype=torch.long).unsqueeze(0).to(device)
    output, _ = model(input_encoded)
    print(output)

[53, 46, 1, 46, 47]
tensor([[[ 6.7816e-01,  4.2309e-01,  1.1232e+00, -5.3999e-01,  4.4096e-01,
          -4.0591e-01, -6.8549e-01,  4.5547e-01, -5.4998e-01, -3.5566e-01,
           4.1656e-01,  5.9446e-01, -8.8711e-01,  6.8487e-01,  7.1445e-02,
           1.5096e-01,  1.1889e+00,  2.3585e-02, -1.7210e+00,  2.4658e-01,
           1.7676e-01,  5.9259e-01,  2.5488e-01,  6.9396e-01, -2.6433e-01,
           3.7112e-01,  8.3249e-02,  9.8497e-02,  7.7958e-01,  5.3378e-01,
           4.0617e-04, -1.1318e+00, -5.6705e-01, -1.8038e-01,  8.3630e-01,
           6.4225e-01,  8.1574e-01,  4.9907e-02,  5.5419e-01,  6.7031e-02,
          -3.6827e-01,  1.1537e+00,  1.4073e+00,  4.0957e-01,  9.2027e-01,
           4.5050e-01, -4.5221e-01, -4.3253e-01, -1.5193e-01, -5.1749e-03,
          -7.1360e-01, -8.9319e-01, -6.8637e-01, -1.0873e+00, -1.2868e-01,
           4.7762e-01, -6.5870e-02,  8.3659e-01, -1.4940e-01, -1.2138e-01,
           3.9251e-01, -2.7171e-02,  1.5801e-01,  8.2163e-01,  2.0967e-01],
    

In [5]:
train_iters = 5000
eval_interval = 500
learning_rate = 2e-4
eval_iters = 100


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(train_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.3655, val loss 4.3617
step 500: train loss 2.1413, val loss 2.1876
step 1000: train loss 1.7369, val loss 1.8744
step 1500: train loss 1.5543, val loss 1.7326
step 2000: train loss 1.4474, val loss 1.6566
step 2500: train loss 1.3773, val loss 1.5984
step 3000: train loss 1.3183, val loss 1.5615
step 3500: train loss 1.2772, val loss 1.5429
step 4000: train loss 1.2418, val loss 1.5288
step 4500: train loss 1.2077, val loss 1.5191


In [6]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(
    decode(
        model.generate(context, max_new_tokens=1000)[0].cpu().tolist()
    )
)


ESTEN:
Now, my lord, to have noble quiet:
On the profitive glorious soul, I shall
For Coriolanus; I'll prove drum: and not
Yet with heavy what I waked by that ord-folling making.

YORK:
I know no broad to publy cousin Bohemy.

CLARENCE:
No mayor: being shall be it.

JULIENT:
I do again. What should stop you than e'er.

DUKE OF YORK:
Fellow none ado! I am falled so deepeped,
How draws too love to some struck man's simple.

GLOUCESTER:
We'll have done, that do'st to this, but all bed,
The whither's trade,--wake't, and wise Half,
They marks infant me queen, and and here
Where I say, and danger and down that tmeet
As bosoes. Was ever was we arms at easy.

GLOUCESTER:
Sweet Bereford nor lord.

LADY ANNE:
Must blender, Sicilia, here he's prince,
What time to-day day, you wonder'd;
And tepted starly not confessor; that even,
It in the tearnest slander to be hold the business,
And by, unruly, wretched the breath of Hereforder,
Lest your such body to speed her old!
Doth gods, to the gods them 